In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q ultralytics
!pip install -q kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d ahmedsa321/bd100k-yolo-conveted --force
!unzip -o bd100k-yolo-conveted.zip -d /content/bdd100k

In [ ]:
!ls /content/bdd100k/dataset_yolo_converted/images
!ls /content/bdd100k/dataset_yolo_converted/images/train | wc -l

In [ ]:
import os
import random
import shutil

base_path = "/content/bdd100k/dataset_yolo_converted"
subset_path = "/content/bdd_subset_10k"

os.makedirs(f"{subset_path}/images/train", exist_ok=True)
os.makedirs(f"{subset_path}/labels/train", exist_ok=True)
os.makedirs(f"{subset_path}/images/val", exist_ok=True)
os.makedirs(f"{subset_path}/labels/val", exist_ok=True)

train_images = os.listdir(f"{base_path}/images/train")
subset_images = random.sample(train_images, 10000)

for img in subset_images:
    shutil.copy(f"{base_path}/images/train/{img}",
                f"{subset_path}/images/train/{img}")

    label_file = img.replace(".jpg", ".txt")
    shutil.copy(f"{base_path}/labels/train/{label_file}",
                f"{subset_path}/labels/train/{label_file}")

shutil.copytree(f"{base_path}/images/val",
                f"{subset_path}/images/val",
                dirs_exist_ok=True)

shutil.copytree(f"{base_path}/labels/val",
                f"{subset_path}/labels/val",
                dirs_exist_ok=True)

print("10K subset created successfully")

In [ ]:
data_yaml = """
path: /content/bdd_subset_10k
train: images/train
val: images/val

names:
  0: area/unknown
  1: bike
  2: bus
  3: car
  4: lane/crosswalk
  5: lane/double other
  6: lane/double white
  7: lane/double yellow
  8: lane/single other
  9: lane/single white
  10: lane/single yellow
  11: motor
  12: person
  13: rider
  14: traffic light
  15: traffic sign
  16: train
  17: truck
"""

with open("/content/data_subset_10k.yaml", "w") as f:
    f.write(data_yaml)

print("YAML created successfully")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="/content/data_subset_10k.yaml",
    epochs=50,
    imgsz=512,
    batch=32,
    device=0,
    amp=True,
    workers=4,
    val=False
)

In [ ]:
metrics = model.val()
print(metrics)

In [ ]:
import pandas as pd

df = pd.read_csv("/content/runs/detect/train/results.csv")

print(df.columns)

In [ ]:
import matplotlib.pyplot as plt

# Training Loss
plt.figure()
plt.plot(df['train/box_loss'], label='Train Box Loss')
plt.plot(df['train/cls_loss'], label='Train Cls Loss')
plt.plot(df['train/dfl_loss'], label='Train DFL Loss')
plt.legend()
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()


# Validation Loss
plt.figure()
plt.plot(df['val/box_loss'], label='Val Box Loss')
plt.plot(df['val/cls_loss'], label='Val Cls Loss')
plt.plot(df['val/dfl_loss'], label='Val DFL Loss')
plt.legend()
plt.title("Validation Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()


# Accuracy Metrics
plt.figure()
plt.plot(df['metrics/mAP50(B)'], label='mAP@50')
plt.plot(df['metrics/mAP50-95(B)'], label='mAP@50-95')
plt.plot(df['metrics/precision(B)'], label='Precision')
plt.plot(df['metrics/recall(B)'], label='Recall')
plt.legend()
plt.title("Model Performance Metrics")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.show()

In [ ]:
results = model.val(conf=0.25, plots=True)
print(results)

In [ ]:
metrics = model.val()
print(metrics)

In [ ]:
metrics = model.val()
print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

In [ ]:
from IPython.display import Image, display

image_path = "/content/bdd_subset_10k/images/val/" + \
             os.listdir("/content/bdd_subset_10k/images/val")[0]

results = model.predict(source=image_path, conf=0.25, save=True)

display(Image(filename=results[0].save_dir + "/" + os.path.basename(image_path)))

In [ ]:
!wget https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4 -O traffic.mp4

In [ ]:
results = model.predict(
    source="/content/traffic.mp4",
    conf=0.25,
    save=True
)

In [ ]:
from IPython.display import Video

Video("/content/runs/detect/predict/traffic.avi", embed=True)

In [ ]:
import time

start = time.time()
model.predict(source="/content/traffic.mp4", save=False)
end = time.time()

print("Total Processing Time:", end - start, "seconds")

In [ ]:
import cv2

cap = cv2.VideoCapture("/content/traffic.mp4")
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

fps = frame_count / (end - start)
print("Approx Video FPS:", fps)

In [ ]:
from collections import Counter

In [ ]:
import os

print("Train Folder Exists:", os.path.exists("/content/runs/detect/train"))
print("Predict Folder Exists:", os.path.exists("/content/runs/detect/predict"))
print("Best Model Exists:", os.path.exists("/content/runs/detect/train/weights/best.pt"))

In [ ]:
import os
import pandas as pd
import cv2

print("===== PROJECT VERIFICATION REPORT =====\n")

# 1️⃣ Training Check
train_exists = os.path.exists("/content/runs/detect/train")
best_model_exists = os.path.exists("/content/runs/detect/train/weights/best.pt")

print("Training Completed:", train_exists)
print("Best Model Saved:", best_model_exists)

# 2️⃣ Results CSV Check
results_csv_exists = os.path.exists("/content/runs/detect/train/results.csv")
print("Results CSV Generated:", results_csv_exists)

if results_csv_exists:
    df = pd.read_csv("/content/runs/detect/train/results.csv")
    print("Epochs Trained:", len(df))

# 3️⃣ Confusion Matrix Check
conf_matrix_exists = os.path.exists("/content/runs/detect/train/confusion_matrix.png")
print("Confusion Matrix Generated:", conf_matrix_exists)

# 4️⃣ Prediction (Video) Check
predict_exists = os.path.exists("/content/runs/detect/predict")
print("Prediction Folder Exists:", predict_exists)

video_output_exists = False
video_path = None

if predict_exists:
    files = os.listdir("/content/runs/detect/predict")
    print("Prediction Files:", files)
    for f in files:
        if f.endswith(".avi") or f.endswith(".mp4"):
            video_output_exists = True
            video_path = f

print("Processed Video Generated:", video_output_exists)

# 5️⃣ FPS Check (Video frame count)
if video_output_exists:
    cap = cv2.VideoCapture(f"/content/runs/detect/predict/{video_path}")
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    print("Processed Video Frame Count:", frame_count)

print("\n===== VERIFICATION COMPLETE =====")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

image_name = list(uploaded.keys())[0]
print("Uploaded Image:", image_name)

In [ ]:
results = model.predict(
    source=image_name,
    conf=0.25,
    save=True
)

In [ ]:
from IPython.display import Image, display

display(Image(filename=f"/content/runs/detect/predict/{image_name}"))

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
video_name = list(uploaded.keys())[0]
print("Uploaded Video:", video_name)

In [ ]:
results = model.predict(
    source=video_name,
    conf=0.25,
    save=True
)

In [ ]:
from IPython.display import Video

# Replace with correct filename if needed
Video("/content/runs/detect/predict/" + video_name.replace(".mp4", ".avi"), embed=True)

In [ ]:
import os
print(os.listdir("/content/runs/detect/predict"))

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt

# 🔹 Use raw string to avoid path issues
video_name = r"/content/Mumbai ka Traffic  Mumbai Traffic  India #shorts - Hungry Cam Raw (360p, h264).mp4"

# Load trained model
model = YOLO("/content/runs/detect/train/weights/best.pt")
class_names = model.names
vehicle_classes = ["car", "bus", "truck", "motorcycle"]

cap = cv2.VideoCapture(video_name)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

heatmap = np.zeros((height, width), dtype=np.float32)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(source=frame, conf=0.25, save=False)

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy()

        for box, cls_id in zip(boxes, classes):
            label = class_names[int(cls_id)]

            if label in vehicle_classes:
                x1, y1, x2, y2 = map(int, box)
                cx = int((x1 + x2) / 2)
                cy = int((y1 + y2) / 2)

                if 0 <= cy < height and 0 <= cx < width:
                    heatmap[cy, cx] += 1

cap.release()

heatmap_blurred = cv2.GaussianBlur(heatmap, (51, 51), 0)

# Normalize
heatmap_norm = cv2.normalize(heatmap_blurred, None, 0, 255, cv2.NORM_MINMAX)
heatmap_uint8 = heatmap_norm.astype(np.uint8)

# color map
colored_heatmap = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

# Save
cv2.imwrite("mumbai_heatmap.jpg", colored_heatmap)

# Display nicely
plt.figure(figsize=(10,6))
plt.imshow(cv2.cvtColor(colored_heatmap, cv2.COLOR_BGR2RGB))
plt.title("Mumbai Traffic Density Heatmap")
plt.axis("off")
plt.show()